# Baseline Churn Model Check + Initial Behavioral Drift Score

Input: `data/features_and_label.parquet` (customer-month rows from notebook 03, label `is_churn`)

Output: drift features + `drift_score` per customer-month

1. Verify that the baseline churn models from 06 beat trivial baselines and are not fooled by leakage
2. Build the first behavioral-drift score & test whether it carries churn signal beyond th existing features


## 1. Setup + Load Data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [ ]:
DATA_PATH     = Path('../data/features_and_label.parquet')
OUT_PATH      = Path('../data/drift_scores.parquet')
TARGET        = 'is_churn'

TEST_FRACTION = 0.20   # same as notebook 06: latest 20% of distinct months are the test set

PURGE_MONTHS  = 3      # = the 90-day label window (~3 months)

K_BASELINE    = 3      # a customer's "normal" = mean of their previous 3 ACTIVE months
CLIP_Z        = 3.0    # cap each standardized drift component at +/-3 so one freak month can't dominate
RANDOM_STATE  = 42